# 08 - Machine Learning Prediction
**IBM Applied Data Science Capstone - SpaceX Falcon 9 First-Stage Landing Prediction**

Author: Pranav Vyankatesh Jagdale

**Goal:** standardise the features, tune four classifiers with GridSearchCV, evaluate them on a hold-out set, and pick a model.

> Run every cell in order. Keep your own outputs - they feed your final slides.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

data = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv")
X = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv")
Y = data['Class'].to_numpy()
print(X.shape, Y.shape)

## 1. Standardise and split

In [ ]:
X = preprocessing.StandardScaler().fit(X).transform(X)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print("train:", X_train.shape, "test:", X_test.shape)
print("baseline (always 'landed'):", round(Y_test.mean(), 3))

## 2. Tune four models

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000),
                            {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}),
    'SVM': (SVC(),
            {'kernel': ['linear', 'rbf', 'poly', 'sigmoid'], 'C': np.logspace(-3, 3, 5), 'gamma': np.logspace(-3, 3, 5)}),
    'Decision Tree': (DecisionTreeClassifier(random_state=2),
                      {'criterion': ['gini', 'entropy'], 'splitter': ['best', 'random'],
                       'max_depth': [2 * n for n in range(1, 10)], 'max_features': ['sqrt', 'log2'],
                       'min_samples_leaf': [1, 2, 4], 'min_samples_split': [2, 5, 10]}),
    'KNN': (KNeighborsClassifier(),
            {'n_neighbors': list(range(1, 11)), 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 'p': [1, 2]}),
}

fitted, rows = {}, []
for name, (model, grid) in models.items():
    gs = GridSearchCV(model, grid, cv=10).fit(X_train, Y_train)
    pred = gs.predict(X_test)
    fitted[name] = gs
    rows.append({'Model': name, 'Best params': gs.best_params_, 'CV acc (train)': round(gs.best_score_, 3),
                 'Test acc': round(accuracy_score(Y_test, pred), 3),
                 'Precision': round(precision_score(Y_test, pred), 3),
                 'Recall': round(recall_score(Y_test, pred), 3),
                 'F1': round(f1_score(Y_test, pred), 3)})
results = pd.DataFrame(rows)
results

## 3. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, gs) in zip(axes, fitted.items()):
    cm = confusion_matrix(Y_test, gs.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cbar=False, ax=ax,
                xticklabels=['did not land', 'landed'], yticklabels=['did not land', 'landed'])
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

## 4. A fairer check: nested cross-validation on all rows
With a very small test set, one prediction moves accuracy a lot. Nested CV tunes inside each fold and scores on rows it never saw.

In [ ]:
outer = StratifiedKFold(10, shuffle=True, random_state=2)
nested = {}
for name, (model, grid) in models.items():
    nested[name] = cross_val_score(GridSearchCV(model, grid, cv=5), X, Y, cv=outer).mean()
results['Nested CV'] = results['Model'].map(lambda m: round(nested[m], 3))
results

In [ ]:
results.drop(columns=['Best params']).set_index('Model')[['CV acc (train)', 'Test acc', 'Nested CV']].plot(kind='bar', figsize=(8, 4), ylim=(0.5, 1.0))
plt.ylabel('Accuracy'); plt.xticks(rotation=0); plt.show()

## Write your own conclusions here
- Which model do you pick and why (accuracy, errors, simplicity)?
- How does the best model compare with the always-'landed' baseline?
- Which error is worse for a company bidding against SpaceX: predicting a landing that fails, or missing a landing that happens?
- Limitations: small sample, features only known after a landing was planned, class imbalance.